In [14]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy import ndimage
import logging
import datetime

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LymphNodeMerger:
    def __init__(self, mri_path, annotation_path, 
                 distance_threshold=2.0,  # in mm
                 contact_area_threshold=10.0,  # in mm²
                 intensity_diff_threshold=0.2,  # relative threshold
                 dilation_radius=1):  # voxels
        """
        Initialize the LymphNodeMerger with the given parameters.
        
        Parameters:
        -----------
        mri_path : str
            Path to the MRI image in NIFTI format
        annotation_path : str
            Path to the annotation image in NIFTI format
        distance_threshold : float
            Maximum distance (in mm) between nodes to be considered for merging
        contact_area_threshold : float
            Minimum contact surface area (in mm²) required for merging
        intensity_diff_threshold : float
            Maximum relative intensity difference allowed between nodes and contact region
        dilation_radius : int
            Radius for dilation operation in voxels
        """
        self.mri_path = mri_path
        self.annotation_path = annotation_path
        self.distance_threshold = distance_threshold
        self.contact_area_threshold = contact_area_threshold
        self.intensity_diff_threshold = intensity_diff_threshold
        self.dilation_radius = dilation_radius
        
        self.mri_image = None
        self.annotation_image = None
        self.spacing = None
        self.num_slides = None
        self.node_labels = None
        self.node_masks = {}
        self.node_stats = {}
        self.adjacency_graph = None
        
        
        logger.info("LymphNodeMerger initialized with parameters:")
        logger.info(f"  Distance threshold: {distance_threshold} mm")
        logger.info(f"  Contact area threshold: {contact_area_threshold} mm²")
        logger.info(f"  Intensity difference threshold: {intensity_diff_threshold}")
        logger.info(f"  Dilation radius: {dilation_radius} voxels")
        
    def load_data(self):
        """Load MRI and annotation data."""
        logger.info(f"Loading MRI image from {self.mri_path}")
        self.mri_image = sitk.ReadImage(self.mri_path)
        
        logger.info(f"Loading annotation image from {self.annotation_path}")
        self.annotation_image = sitk.ReadImage(self.annotation_path)
        
        # Ensure same coordinate system
        if not self._check_coordinate_match():
            logger.warning("MRI and annotation images might not be in the same coordinate system!")
        
        self.spacing = self.mri_image.GetSpacing()
        logger.info(f"Image spacing: {self.spacing}")
        
        # Extract node labels
        np_annotation = sitk.GetArrayFromImage(self.annotation_image)
        self.node_labels = np.unique(np_annotation)
        self.node_labels = self.node_labels[self.node_labels > 0]  # Remove background
        
        logger.info(f"Found {len(self.node_labels)} lymph node annotations with labels: {self.node_labels}")
        
        # Create individual masks for each node
        self._create_node_masks()
        
        return self
    
    def _check_coordinate_match(self):
        """Check if MRI and annotation images have matching coordinate systems."""
        mri_size = self.mri_image.GetSize()
        anno_size = self.annotation_image.GetSize()
        mri_spacing = self.mri_image.GetSpacing()
        anno_spacing = self.annotation_image.GetSpacing()
        mri_origin = self.mri_image.GetOrigin()
        anno_origin = self.annotation_image.GetOrigin()
        
        size_match = mri_size == anno_size
        spacing_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_spacing, anno_spacing))
        origin_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_origin, anno_origin))

        self.num_slides = anno_size[2]
        
        logger.info(f"Size match: {size_match}, Spacing match: {spacing_match}, Origin match: {origin_match}")
        logger.info(f"MRI spacing: {mri_spacing}, Anno spacing: {anno_spacing}")
        logger.info(f"xyz: {anno_size}, num_slides: {self.num_slides}")
        
        return size_match and spacing_match and origin_match
    
    def _create_node_masks(self):
        """Create binary masks for each lymph node."""
        for label in self.node_labels:
            logger.info(f"Creating mask for node {label}")
            
            # Create binary mask for this node
            node_mask = sitk.Equal(self.annotation_image, int(label))
            self.node_masks[label] = node_mask
            
            # Calculate basic statistics for this node
            np_mri = sitk.GetArrayFromImage(self.mri_image)
            np_mask = sitk.GetArrayFromImage(node_mask)
            node_voxels = np_mri[np_mask > 0]
            
            if len(node_voxels) > 0:
                self.node_stats[label] = {
                    'mean_intensity': np.mean(node_voxels),
                    'std_intensity': np.std(node_voxels),
                    'volume_mm3': np.sum(np_mask) * np.prod(self.spacing),
                    'voxel_count': np.sum(np_mask)
                }
                logger.info(f"  Node {label} stats: {self.node_stats[label]}")
            else:
                logger.warning(f"  Node {label} has no voxels!")
    
    def analyze_node_pairs(self):
        """Analyze all pairs of nodes to determine potential merges."""
        logger.info("Starting node pair analysis")
        
        # Create a graph where nodes are lymph nodes and edges represent potential merges
        self.adjacency_graph = nx.Graph()
        for label in self.node_labels:
            self.adjacency_graph.add_node(label)
        
        # Analyze all unordered pairs
        node_pairs = [(a, b) for i, a in enumerate(self.node_labels) 
                     for b in self.node_labels[i+1:]]
        
        logger.info(f"Analyzing {len(node_pairs)} node pairs")
        
        for node_a, node_b in node_pairs:
            logger.info(f"Analyzing node pair ({node_a}, {node_b})")
            
            # Check minimum distance
            min_distance = self._calculate_min_distance(node_a, node_b)
            logger.info(f"  Minimum distance: {min_distance} mm")
            
            # Only proceed if distance is within threshold
            if min_distance > self.distance_threshold:
                logger.info("  Distance exceeds threshold, skipping further analysis")
                continue
            
            # Calculate contact surface area after dilation
            contact_area, contact_mask, dilated_a, dilated_b = self._calculate_contact_area(node_a, node_b)

            logger.info(f"  Contact surface area: {contact_area} mm²")
            
            if contact_area < self.contact_area_threshold:
                logger.info("  Contact area below threshold, skipping further analysis")
                continue
            
            # Check intensity similarity
            is_similar, similarity_score = self._check_intensity_similarity(node_a, node_b, contact_mask)
            logger.info(f"  Intensity similarity: {is_similar} (score: {similarity_score:.3f})")
            
            if is_similar:
                logger.info(f"  Adding edge between nodes {node_a} and {node_b}")
                self.adjacency_graph.add_edge(node_a, node_b, 
                                             distance=min_distance,
                                             contact_area=contact_area,
                                             similarity=similarity_score)
        
        return self
    
    def _calculate_min_distance(self, node_a, node_b):
        """Calculate minimum surface-to-surface distance between two nodes."""
        ## Get coordinates of mask voxels
        mask_a = sitk.GetArrayFromImage(self.node_masks[node_a]) > 0
        mask_b = sitk.GetArrayFromImage(self.node_masks[node_b]) > 0
        
        # If either mask is empty, return infinity
        if not np.any(mask_a) or not np.any(mask_b):
            return np.inf
        
        # Get coordinates of boundary voxels
        # A voxel is on the boundary if it's part of the mask and has at least one neighbor that isn't
        struct = ndimage.generate_binary_structure(3, 1)  # 3D connectivity
        eroded_a = ndimage.binary_erosion(mask_a, struct)
        boundary_a = mask_a & ~eroded_a
        
        eroded_b = ndimage.binary_erosion(mask_b, struct)
        boundary_b = mask_b & ~eroded_b
        
        # Get indices of boundary voxels
        boundary_a_indices = np.argwhere(boundary_a)
        boundary_b_indices = np.argwhere(boundary_b)
        
        # Convert indices to physical coordinates using spacing
        spacing_xyz = self.spacing[::-1]  # Convert from ITK to numpy format
        boundary_a_coords = boundary_a_indices * spacing_xyz
        boundary_b_coords = boundary_b_indices * spacing_xyz
        
        # Calculate minimum distance using KDTree for efficiency
        from scipy.spatial import KDTree
        tree_a = KDTree(boundary_a_coords)
        tree_b = KDTree(boundary_b_coords)
        
        # Find minimum distance from A to B
        min_dist_a_to_b = np.min(tree_a.query(boundary_b_coords)[0])
        
        # Find minimum distance from B to A
        min_dist_b_to_a = np.min(tree_b.query(boundary_a_coords)[0])
        
        return min(min_dist_a_to_b, min_dist_b_to_a)
    
    
    def _calculate_contact_area(self, node_a, node_b):
        """Calculate contact surface area between two nodes after dilation."""

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        # Get original masks
        original_a = self.node_masks[node_a]
        original_b = self.node_masks[node_b]
        
        # Dilate both masks
        dilated_a = self._dilate_mask(original_a)
        dilated_b = self._dilate_mask(original_b)

        filename_a_orig = f"{timestamp}_node_{node_a}_original.nii.gz"
        filename_a_dil = f"{timestamp}_node_{node_a}_dilated.nii.gz"
        
        sitk.WriteImage(original_a, filename_a_orig)
        logger.info(f"Saved {filename_a_orig}")
        sitk.WriteImage(dilated_a, filename_a_dil)
        logger.info(f"Saved {filename_a_dil}")

        filename_b_orig = f"{timestamp}_node_{node_b}_original.nii.gz"
        filename_b_dil = f"{timestamp}_node_{node_b}_dilated.nii.gz"
        
        sitk.WriteImage(original_b, filename_b_orig)
        logger.info(f"Saved {filename_b_orig}")
        sitk.WriteImage(dilated_b, filename_b_dil)
        logger.info(f"Saved {filename_b_dil}")
        
        # Find intersection of dilated masks
        intersection = sitk.And(dilated_a, dilated_b)
        
        # Remove original masks from intersection to get only the contact region
        original_union = sitk.Or(original_a, original_b)
        contact_region = sitk.And(intersection, sitk.Not(original_union))
        
        # Calculate surface area
        np_contact = sitk.GetArrayFromImage(contact_region)
        voxel_volume = np.prod(self.spacing)
        contact_voxels = np.sum(np_contact)
        
        # Log contact region information
        logger.info(f"  Contact region: {contact_voxels} voxels")
        
        # Estimate surface area - this is an approximation
        # For a more accurate calculation, we would need to count boundary voxels
        contact_area = contact_voxels * voxel_volume**(2/3)
        
        return contact_area, contact_region, dilated_a, dilated_b
    
    
    def _dilate_mask(self, mask: sitk.Image) -> sitk.Image:
        """
        Dilate a binary mask with SimpleITK only.
        The operation is applied to every axial (x–y) slice individually.

        Parameters
        mask : sitk.Image
            3-D binary image (0 background, >0 foreground).

        Returns
        sitk.Image
            Dilated 3-D mask (uint8, 0/1) with the same meta-data as the input.
        """
        # 1. Ensure the mask is strictly 0/1 (cast to uint8 for dilation).
        original_mask = mask
        binary_mask   = sitk.Cast(mask > 0, sitk.sitkUInt8)

        original_count = int(sitk.GetArrayViewFromImage(binary_mask).sum())

        # 2. Prepare the 2-D extractor and dilater.
        size  = list(binary_mask.GetSize())          # [x, y, z]
        depth = size[2]

        extractor = sitk.ExtractImageFilter()
        extractor.SetSize([size[0], size[1], 0])     # 2-D slice size

        dilater = sitk.BinaryDilateImageFilter()
        dilater.SetForegroundValue(1)
        dilater.SetBackgroundValue(0)
        dilater.SetKernelType(sitk.sitkBall)         # circular kernel in 2-D
        dilater.SetKernelRadius(self.dilation_radius)

        # 3. Dilate every slice and collect the results.
        dilated_slices = []
        for z in range(depth):
            extractor.SetIndex([0, 0, z])
            slice2d        = extractor.Execute(binary_mask)
            dilated_slice  = dilater.Execute(slice2d)
            dilated_slices.append(dilated_slice)

        # 4. Stack the 2-D slices back into a 3-D volume.
        dilated_volume = sitk.JoinSeries(dilated_slices)
        # JoinSeries already returns a 3-D image (x, y, sliceId),
        # so nothing else is needed except restoring the original meta-data.
        dilated_volume.CopyInformation(original_mask)

        # 5. Book-keeping / logging.
        dilated_count = int(sitk.GetArrayViewFromImage(dilated_volume).sum())
        logger.info(
            f"  Dilation: {original_count} voxels -> {dilated_count} voxels "
            f"(+{dilated_count - original_count}, "
            f"{dilated_count / max(original_count, 1):.2f}x)"
        )

        return dilated_volume
    
    
    def _check_intensity_similarity(self, node_a, node_b, contact_mask):
        """Check if the intensity in the contact region is similar to the nodes."""
        # Get intensity values
        np_mri = sitk.GetArrayFromImage(self.mri_image)
        np_contact = sitk.GetArrayFromImage(contact_mask)
        
        # Extract intensities for each region
        if np.sum(np_contact) == 0:
            logger.warning("  Contact region is empty")
            return False, 0
            
        contact_intensities = np_mri[np_contact > 0]
        mean_contact = np.mean(contact_intensities)
        
        # Compare with node intensities
        mean_a = self.node_stats[node_a]['mean_intensity']
        mean_b = self.node_stats[node_b]['mean_intensity']
        
        # Calculate weighted average of node intensities
        weighted_mean = (mean_a * self.node_stats[node_a]['voxel_count'] + 
                         mean_b * self.node_stats[node_b]['voxel_count']) / \
                        (self.node_stats[node_a]['voxel_count'] + 
                         self.node_stats[node_b]['voxel_count'])
        
        # Calculate relative difference
        rel_diff = abs(mean_contact - weighted_mean) / weighted_mean
        
        # Check if intensity difference is within threshold
        is_similar = rel_diff <= self.intensity_diff_threshold
        
        return is_similar, 1 - rel_diff  # Convert to similarity score (0-1)
    
    def identify_matted_nodes(self):
        """Identify groups of matted nodes using connected components."""
        if self.adjacency_graph is None:
            logger.error("Node pair analysis has not been run yet!")
            return None
        
        # Find connected components in the adjacency graph
        connected_components = list(nx.connected_components(self.adjacency_graph))
        
        logger.info(f"Found {len(connected_components)} matted node groups:")
        for i, component in enumerate(connected_components):
            logger.info(f"  Group {i+1}: {component}")
        
        return connected_components
    
    def merge_annotations(self):
        """Merge annotations based on identified matted node groups."""
        matted_groups = self.identify_matted_nodes()
        
        if not matted_groups:
            logger.info("No matted node groups to merge")
            return self.annotation_image
        
        # Create a new label image by copying the original
        merged_annotation = sitk.Cast(self.annotation_image, self.annotation_image.GetPixelID())
        
        # Process each matted group
        for i, group in enumerate(matted_groups):
            if len(group) <= 1:
                logger.info(f"Skipping group {i+1} as it contains only one node")
                continue
                
            logger.info(f"Merging group {i+1}: {group}")
            
            # Create a new label for this group (use the minimum label in the group)
            new_label = min(group)
            
            # Create a mask for the entire group
            group_mask = sitk.Image(self.annotation_image.GetSize(), sitk.sitkUInt8)
            group_mask.CopyInformation(self.annotation_image)
            
            # Union all node masks in this group
            for node_label in group:
                if node_label != new_label:  # Skip the new label as it will stay the same
                    # Create a binary mask for this node
                    temp_mask = sitk.Equal(self.annotation_image, int(node_label))
                    
                    # Add to group mask
                    group_mask = sitk.Or(group_mask, temp_mask)
                    
                    # Remove the original node from the merged annotation by setting it to 0
                    # This is equivalent to: merged_annotation = sitk.Where(temp_mask, 0, merged_annotation)
                    zero_image = sitk.Image(merged_annotation.GetSize(), merged_annotation.GetPixelID())
                    zero_image.CopyInformation(merged_annotation)
                    
                    # Multiply inverted mask with merged annotation (sets masked areas to 0)
                    inverted_mask = sitk.Not(temp_mask)
                    merged_annotation = sitk.Multiply(
                        merged_annotation, 
                        sitk.Cast(inverted_mask, merged_annotation.GetPixelID())
                    )
            
            # Add the new label to the group areas
            # This is equivalent to: merged_annotation = sitk.Where(group_mask, new_label, merged_annotation)
            # First, create an image filled with the new label
            label_image = sitk.Image(merged_annotation.GetSize(), merged_annotation.GetPixelID())
            label_image.CopyInformation(merged_annotation)
            label_image = sitk.Add(label_image, float(new_label))
            
            # Then, use masking to combine: (mask * label_image) + ((1-mask) * merged_annotation)
            merged_annotation = sitk.Add(
                sitk.Multiply(
                    sitk.Cast(group_mask, merged_annotation.GetPixelID()),
                    label_image
                ),
                sitk.Multiply(
                    sitk.Cast(sitk.Not(group_mask), merged_annotation.GetPixelID()),
                    merged_annotation
                )
            )
        
        logger.info("Annotation merging completed")
        return merged_annotation
    
        
    # # If there are matted groups, visualize the graph
    # if hasattr(self, 'adjacency_graph') and len(self.adjacency_graph.edges) > 0:
    #     plt.figure(figsize=(10, 8))
    #     pos = nx.spring_layout(self.adjacency_graph)
        
    #     # Draw nodes
    #     nx.draw_networkx_nodes(self.adjacency_graph, pos, 
    #                             node_color='lightblue', 
    #                             node_size=500)
        
    #     # Draw edges with weights based on similarity
    #     edge_weights = [self.adjacency_graph[u][v].get('similarity', 0.5) 
    #                     for u, v in self.adjacency_graph.edges()]
        
    #     nx.draw_networkx_edges(self.adjacency_graph, pos, 
    #                             width=[w*3 for w in edge_weights],
    #                             alpha=0.7)
        
    #     # Draw labels
    #     nx.draw_networkx_labels(self.adjacency_graph, pos, 
    #                             font_size=12, 
    #                             font_family='sans-serif')
        
    #     plt.title("Node Adjacency Graph (Edge Width = Similarity)")
    #     plt.axis('off')
    #     plt.tight_layout()
    #     plt.show()

    
    def save_results(self, output_path):
        """Save the merged annotation to file."""
        if not hasattr(self, 'merged_annotation'):
            logger.info("Running merge_annotations first")
            self.merged_annotation = self.merge_annotations()
            
        logger.info(f"Saving merged annotations to {output_path}")
        sitk.WriteImage(self.merged_annotation, output_path)
        
        return self
    
    def run_pipeline(self, output_path=None, slice_idx=None):
        """Run the complete analysis pipeline."""
        (self.load_data()
             .analyze_node_pairs())
        
        self.merged_annotation = self.merge_annotations()
            
        if output_path:
            self.save_results(output_path)
            
        return self

if __name__ == "__main__":
    mri_path = "data/raw/images/1095-T2_FS_TRA+301.nii.gz"
    annotation_path = "data/raw/labels/1095-T2_FS_TRA+301.nii.gz"
    output_path = "test.nii.gz"
    
    merger = LymphNodeMerger(
        mri_path=mri_path,
        annotation_path=annotation_path,
        distance_threshold=2.0,
        contact_area_threshold=10.0,
        intensity_diff_threshold=0.2,
        dilation_radius=1,
    )
    
    merger.run_pipeline(output_path=output_path)
    

2025-07-11 10:57:52,463 - INFO - LymphNodeMerger initialized with parameters:
2025-07-11 10:57:52,464 - INFO -   Distance threshold: 2.0 mm
2025-07-11 10:57:52,465 - INFO -   Contact area threshold: 10.0 mm²
2025-07-11 10:57:52,466 - INFO -   Intensity difference threshold: 0.2
2025-07-11 10:57:52,467 - INFO -   Dilation radius: 1 voxels
2025-07-11 10:57:52,476 - INFO - Loading MRI image from data/raw/images/1095-T2_FS_TRA+301.nii.gz
2025-07-11 10:57:52,786 - INFO - Loading annotation image from data/raw/labels/1095-T2_FS_TRA+301.nii.gz
2025-07-11 10:57:52,828 - INFO - Size match: True, Spacing match: True, Origin match: True
2025-07-11 10:57:52,829 - INFO - MRI spacing: (0.44920000433921814, 0.44920000433921814, 4.0), Anno spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-11 10:57:52,830 - INFO - xyz: (512, 512, 30), num_slides: 30
2025-07-11 10:57:52,831 - INFO - Image spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-11 10:57:52,935 - INFO - Found 7 lym